In [1]:
!pip install -q gdown
!gdown "https://drive.google.com/uc?id=1dZ984LG7WsuzCi91gE4svj1dZ2zZlpTk" -O dataset.csv

Downloading...
From: https://drive.google.com/uc?id=1dZ984LG7WsuzCi91gE4svj1dZ2zZlpTk
To: /content/dataset.csv
100% 50.0M/50.0M [00:01<00:00, 41.3MB/s]


In [2]:
import pandas as pd
import re


In [3]:
df = pd.read_csv('/content/dataset.csv')

In [8]:
df

,Source,Brand,Specialization,Description,Adress,Date,Score,Comment
0,DGIS,Sushi Holl,Ресторан доставки,Дополнительное описание отсутсвует,"​Пискунова, 146, Иркутск",02.08.2023,5.0,Вкусно очень
1,DGIS,Sushi Holl,Ресторан доставки,Дополнительное описание отсутсвует,"​Пискунова, 146, Иркутск",30.07.2023,4.0,Заказывали роллы все свежее вкусное только тем...
2,DGIS,Sushi Holl,Ресторан доставки,Дополнительное описание отсутсвует,"​Пискунова, 146, Иркутск",28.07.2023,5.0,цена качество и скорость топ советую
3,DGIS,Sushi Holl,Ресторан доставки,Дополнительное описание отсутсвует,"​Пискунова, 146, Иркутск",27.07.2023,5.0,Моя любимая доставка сушей в Иркутске незнаю ...
4,DGIS,Sushi Holl,Ресторан доставки,Дополнительное описание отсутсвует,"​Пискунова, 146, Иркутск",12.07.2023,1.0,В шоке от роллов такое чувство что просто пос...
...,...,...,...,...,...,...,...,...
94064,DGIS,Фруктовый мир,Продуктовый магазин,Дополнительное описание отсутсвует,"​Байкальская, 422, Иркутск",20.06.2022,5.0,Всем привет принимаем заказ по Иркутску достав...
94065,DGIS,Продуктовый магазин,Продуктовые магазины,Дополнительное описание отсутсвует,"​Красного Восстания, 20, Иркутск",02.08.2023,5.0,Доброжелательный персонал
94066,DGIS,Продуктовый магазин,Продуктовые магазины,Дополнительное описание отсутсвует,"​Красного Восстания, 20, Иркутск",12.07.2022,5.0,Хороший и продавец вежливый редко где сей час ...
94067,DGIS,Продуктовый магазин,Продуктовые магазины,Дополнительное описание отсутсвует,"​Красного Восстания, 20, Иркутск",20.12.2020,5.0,топ магаз спасли мои студенческие времена


In [15]:
print('Исходный размер датасета:')
print(df.shape)

print('\nИсходное количество уникальных значений:')
unique_before = df.nunique(dropna=False)
print(unique_before)

print('\nКоличество пропусков в каждом столбце:')
missing_count = df.isna().sum()
print(missing_count)

Исходный размер датасета:
(94069, 8)

Исходное количество уникальных значений:
Source                1
Brand               500
Specialization      179
Description           1
Adress              624
Date               2770
Score                 6
Comment           87441
dtype: int64

Количество пропусков в каждом столбце:
Source               0
Brand                0
Specialization       0
Description          0
Adress              50
Date                69
Score               69
Comment           5286
dtype: int64


In [17]:
#Очистка текстовых значений
def clean_text_value(value):
    if pd.isna(value):
        return value

    value = str(value)
    value = value.replace('\xa0', ' ')
    value = value.replace('\u200b', '')
    value = value.replace('\ufeff', '')
    value = value.strip()
    value = ' '.join(value.split())
    return value

In [18]:
#Удаление полных дубликатов
duplicates_count = df.duplicated().sum()
df = df.drop_duplicates()
print('\nРазмер датасета после удаления полных дубликатов:')
print(df.shape)


Размер датасета после удаления полных дубликатов:
(91719, 8)


In [19]:
# Очистка столбцов
df['Source'] = df['Source'].apply(clean_text_value)


df['Brand'] = df['Brand'].apply(clean_text_value)
empty_brand_count = df['Brand'].isna().sum() + (df['Brand'] == '').sum()
df = df[df['Brand'].notna()].copy()
df = df[df['Brand'] != ''].copy()


df['Specialization'] = df['Specialization'].apply(clean_text_value)
empty_specialization_count = df['Specialization'].isna().sum() + (df['Specialization'] == '').sum()
df = df[df['Specialization'].notna()].copy()
df = df[df['Specialization'] != ''].copy()


df['Description'] = df['Description'].apply(clean_text_value)


df['Adress'] = df['Adress'].apply(clean_text_value)
df['Adress'] = df['Adress'].str.replace(r'\d+\s*филиал[а-я]*', '', regex=True)
df['Adress'] = df['Adress'].apply(clean_text_value)


df['Date'] = df['Date'].apply(clean_text_value)
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y', errors='coerce')
invalid_date_count = df['Date'].isna().sum()
df = df[df['Date'].notna()].copy()


df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
invalid_score_count = df['Score'].isna().sum()
df = df[df['Score'].notna()].copy()
wrong_score_count = len(df[~df['Score'].between(1, 5)])
df = df[df['Score'].between(1, 5)].copy()
df['Score'] = df['Score'].astype(int)


df['Comment'] = df['Comment'].apply(clean_text_value)
df['Comment_clean'] = df['Comment'].fillna('').str.lower()
df['Comment_clean'] = df['Comment_clean'].str.replace(r'[^\w\sа-яА-ЯёЁ]', ' ', regex=True)
df['Comment_clean'] = df['Comment_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()


df_text = df[df['Comment_clean'].notna()].copy()
df_text = df_text[df_text['Comment_clean'] != ''].copy()

In [20]:
df.to_csv('/content/dataset_clean.csv', index=False)
df_text.to_csv('/content/dataset_text_clean.csv', index=False)

print('\nОчищенный датасет сохранён в файл:')
print('/content/dataset_clean.csv')

print('\nДатасет для текстового анализа сохранён в файл:')
print('/content/dataset_text_clean.csv')


Очищенный датасет сохранён в файл:
/content/dataset_clean.csv

Датасет для текстового анализа сохранён в файл:
/content/dataset_text_clean.csv


In [22]:
print('Новый размер датасета:')
print(df.shape)

print('\nНовое количество уникальных значений:')
unique_before = df.nunique(dropna=False)
print(unique_before)

print('\nКоличество пропусков в каждом столбце:')
missing_count = df.isna().sum()
print(missing_count)

Новый размер датасета:
(91653, 9)

Новое количество уникальных значений:
Source                1
Brand               463
Specialization      168
Description           1
Adress              474
Date               2768
Score                 5
Comment           86882
Comment_clean     86004
dtype: int64

Количество пропусков в каждом столбце:
Source               0
Brand                0
Specialization       0
Description          0
Adress               8
Date                 0
Score                0
Comment           3233
Comment_clean        0
dtype: int64


Comment_clean - копия Comment, но подготовленная для анализа

dataset_text_clean.csv - файл для анализа отзывов